# Notebook 8: Physical Realizability (EOT)

Evaluate physical realizability of attacks using Expectation Over Transformations.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from helpers import load_scenario, get_all_detections, get_ground_truth, get_ownship

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)
from attacks.camera_attacks import CameraAdversarialAttacker, AttackType
from attacks.physical_eot import MaritimeEOT, MaritimeEnvironmentParams

## 8.1 Load Data

In [ ]:
SCENARIO = 'scenario2'
loader = load_scenario(SCENARIO)
detections = get_all_detections(loader)
ir_detections = detections[3]

print('IR Camera:', len(ir_detections), 'detections')

## 8.2 Initialize Physical Model

In [ ]:
conditions = {
    'Calm': {'wave_height': 0.5, 'rain_rate': 0.0, 'fog_visibility': 10000, 'sun_glint_angle': 90},
    'Moderate Waves': {'wave_height': 2.0, 'rain_rate': 0.0, 'fog_visibility': 10000, 'sun_glint_angle': 90},
    'Heavy Rain': {'wave_height': 0.5, 'rain_rate': 10.0, 'fog_visibility': 10000, 'sun_glint_angle': 90},
    'Fog': {'wave_height': 0.5, 'rain_rate': 0.0, 'fog_visibility': 500, 'sun_glint_angle': 90},
    'Sun Glint': {'wave_height': 0.5, 'rain_rate': 0.0, 'fog_visibility': 10000, 'sun_glint_angle': 10},
}

models = {name: MaritimeEOT(**params) for name, params in conditions.items()}
print('Initialized', len(models), 'environment conditions')

## 8.3 Generate Attack

In [ ]:
attacker = CameraAdversarialAttacker(epsilon=0.05)
attacked = attacker.attack_detections(ir_detections.copy(), AttackType.FGSM, sensor_id=3)

perturbation = attacked['bearing'].dropna().values - ir_detections['bearing'].dropna().values
print('Perturbation range: [', round(perturbation.min(), 4), ',', round(perturbation.max(), 4), '] rad')
print('Mean perturbation:', round(np.mean(np.abs(perturbation)), 4), 'rad')

## 8.4 Evaluate Realizability Scores

In [ ]:
realizability_results = {}

for condition_name, model in models.items():
    scores = []
    valid_dets = ir_detections.dropna(subset=['x_piren', 'y_piren', 'bearing'])
    for i in range(min(100, len(valid_dets))):
        det = {
            'x_piren': valid_dets.iloc[i]['x_piren'],
            'y_piren': valid_dets.iloc[i]['y_piren'],
            'time': valid_dets.iloc[i]['time']
        }
        score = model.transform_detection(det, perturbation[i % len(perturbation)])
        scores.append(score)
    
    realizability_results[condition_name] = {
        'mean': np.mean(scores),
        'std': np.std(scores),
        'min': np.min(scores),
        'max': np.max(scores)
    }
    print(condition_name + ': mean=' + str(round(np.mean(scores), 3)) + ', std=' + str(round(np.std(scores), 3)))

## 8.5 Visualize Realizability by Condition

In [ ]:
conditions_names = list(realizability_results.keys())
means = [realizability_results[c]['mean'] for c in conditions_names]
stds = [realizability_results[c]['std'] for c in conditions_names]

plt.figure(figsize=(12, 6))
plt.bar(conditions_names, means, yerr=stds, capsize=5, 
        color=['green', 'yellow', 'orange', 'gray', 'red'], alpha=0.7)
plt.axhline(y=0.5, color='k', linestyle='--', label='Threshold (0.5)')
plt.ylabel('Realizability Score')
plt.title('Physical Realizability of FGSM Attack Under Different Conditions')
plt.ylim(0, 1)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8.6 EOT with Multiple Samples

In [ ]:
sample_counts = [1, 5, 10, 20, 50]
eot_results = []

model = MaritimeEOT(MaritimeEnvironmentParams(wave_height=1.0, rain_rate=5.0, fog_visibility=2000)
valid_dets = ir_detections.dropna(subset=['x_piren', 'y_piren', 'bearing'])

for n_samples in sample_counts:
    scores = []
    for i in range(min(50, len(valid_dets))):
        det = {
            'x_piren': valid_dets.iloc[i]['x_piren'],
            'y_piren': valid_dets.iloc[i]['y_piren'],
            'time': valid_dets.iloc[i]['time']
        }
        sample_scores = [model.transform_detection(det, perturbation[i % len(perturbation)]) for _ in range(n_samples)]
        scores.append(np.mean(sample_scores))
    
    eot_results.append({'n_samples': n_samples, 'mean_score': np.mean(scores)})
    print('n_samples=' + str(n_samples) + ': mean realizability=' + str(round(np.mean(scores), 3)))

plt.figure(figsize=(10, 6))
plt.plot([r['n_samples'] for r in eot_results], [r['mean_score'] for r in eot_results], 
         'bo-', linewidth=2, markersize=10)
plt.xlabel('Number of EOT Samples')
plt.ylabel('Mean Realizability Score')
plt.title('EOT Convergence: Realizability vs Sample Count')
plt.grid(True)
plt.show()

## 8.7 Attack Success vs Realizability Trade-off

In [ ]:
epsilons = np.linspace(0.01, 0.2, 10)
model = MaritimeEOT(MaritimeEnvironmentParams(wave_height=1.0, rain_rate=2.0)
valid_dets = ir_detections.dropna(subset=['x_piren', 'y_piren', 'bearing'])

trade_off = []

for eps in epsilons:
    att = CameraAdversarialAttacker(epsilon=eps)
    attacked = att.attack_detections(ir_detections.copy(), AttackType.FGSM, sensor_id=3)
    pert = attacked['bearing'].dropna().values - ir_detections['bearing'].dropna().values
    
    attack_success = np.mean(np.abs(pert))
    
    scores = []
    for i in range(min(50, len(valid_dets))):
        det = {
            'x_piren': valid_dets.iloc[i]['x_piren'],
            'y_piren': valid_dets.iloc[i]['y_piren'],
            'time': valid_dets.iloc[i]['time']
        }
        scores.append(model.transform_detection(det, pert[i % len(pert)]))
    
    realizability = np.mean(scores)
    trade_off.append({'epsilon': eps, 'attack_success': attack_success, 'realizability': realizability})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot([t['epsilon'] for t in trade_off], [t['attack_success'] for t in trade_off], 
         'ro-', linewidth=2, markersize=8)
ax1.set_xlabel('Epsilon')
ax1.set_ylabel('Attack Success (rad)')
ax1.set_title('Attack Success vs Epsilon')
ax1.grid(True)

ax2.plot([t['epsilon'] for t in trade_off], [t['realizability'] for t in trade_off], 
         'bo-', linewidth=2, markersize=8)
ax2.set_xlabel('Epsilon')
ax2.set_ylabel('Realizability Score')
ax2.set_title('Realizability vs Epsilon')
ax2.grid(True)

plt.suptitle('Attack Success vs Physical Realizability Trade-off', fontsize=14)
plt.tight_layout()
plt.show()